In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abhisheksjha/time-series-air-quality-data-of-india-2010-2023")

print("Path to dataset files:", path)

In [ ]:
# Correlation matrix (fast)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

numeric = df.select_dtypes(include=[np.number])        # numeric-only
corr = numeric.corr()                                  # pairwise correlations
plt.figure(figsize=(12,10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix (numeric features)")
plt.tight_layout()
plt.savefig("corr_matrix.png", dpi=200)
plt.show()

In [ ]:
# Most important by absolute correlation and by Lasso coefficients (fast)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

target_col = "PM2.5 (ug/m3)"   # adjust if needed

# Prepare X, y (numeric features only)
X = df.select_dtypes(include=[np.number]).copy()
if target_col not in X.columns:
    raise ValueError(f"target_col {target_col} not in numeric columns")
y = X.pop(target_col)

# Impute simple mean (quick)
X = X.fillna(X.mean())
y = y.fillna(y.mean())

# 1) highest absolute correlation with target
corr_with_target = X.corrwith(y).abs().sort_values(ascending=False)
most_corr_feature = corr_with_target.index[0]
print("Most correlated feature (abs corr):", most_corr_feature, corr_with_target.iloc[0])

# 2) quick Lasso for importance (standardize first)
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
lasso = Lasso(alpha=0.1, random_state=0, max_iter=5000)
lasso.fit(Xs, y)
coef = pd.Series(np.abs(lasso.coef_), index=X.columns).sort_values(ascending=False)
most_lasso = coef.idxmax()
print("Most important by Lasso abs(coef):", most_lasso, coef.iloc[0])

In [ ]:
# Least important by correlation and Lasso
# Using same X and y from previous snippet

# 1) smallest absolute correlation
least_corr_feature = corr_with_target.index[-1]
print("Least correlated feature (abs corr):", least_corr_feature, corr_with_target.iloc[-1])

# 2) features with zero (or near-zero) Lasso coefficients
zero_lasso = coef[coef <= 1e-6].index.tolist()
print("Features with near-zero Lasso coefficient (likely least important):", zero_lasso[:10])

In [ ]:
# SelectKBest (F-regression) and mutual information (fast)
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

k = 10  # how many features to select
skb = SelectKBest(score_func=f_regression, k=min(k, X.shape[1]))
skb.fit(Xs, y)
selected_skb = X.columns[skb.get_support()].tolist()
print("SelectKBest (F):", selected_skb)

mi = mutual_info_regression(Xs, y, random_state=0)
mi_sr = pd.Series(mi, index=X.columns).sort_values(ascending=False)
print("Top mutual-info features:", mi_sr.head(k).index.tolist())

In [ ]:
# SequentialFeatureSelector (forward & backward) using a fast linear estimator
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import Ridge

est = Ridge(alpha=1.0)   # cheap estimator vs RandomForest
n_select = 5

sfs_forward = SequentialFeatureSelector(est, n_features_to_select=n_select, direction="forward", cv=3, n_jobs=-1)
sfs_forward.fit(Xs, y)
forward_features = X.columns[sfs_forward.get_support()].tolist()
print("Forward SFS features:", forward_features)

sfs_backward = SequentialFeatureSelector(est, n_features_to_select=n_select, direction="backward", cv=3, n_jobs=-1)
sfs_backward.fit(Xs, y)
backward_features = X.columns[sfs_backward.get_support()].tolist()
print("Backward SFS features:", backward_features)

In [ ]:
# PCA to reduce dimensionality and examine explained variance
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
Xs = scaler.fit_transform(X)   # reuse X numeric-imputed

pca = PCA()
pca.fit(Xs)
explained = pca.explained_variance_ratio_
cumsum = explained.cumsum()
components_95 = int((cumsum >= 0.95).argmax()) + 1 if (cumsum >= 0.95).any() else len(explained)
print("Components needed for 95% variance:", components_95)

# quick plot
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(np.arange(1,len(cumsum)+1), cumsum, marker="o")
plt.axhline(0.95, color="r", linestyle="--")
plt.xlabel("Number of components"); plt.ylabel("Cumulative explained variance")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Transform into reduced PCA feature matrix:
pca_reduced = PCA(n_components=components_95)
X_pca = pca_reduced.fit_transform(Xs)
print("X_pca shape:", X_pca.shape)

In [ ]:
# Example feature engineering snippets (non-destructive, fast)
# operate on df (original)
df_fe = df.copy()

# 1) polynomial (square) for first few numeric cols
num_cols = df_fe.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols[:5]:
    df_fe[f"{c}_sq"] = df_fe[c] ** 2

# 2) interactions (example NO * NO2) if present
if "NO (ug/m3)" in df_fe.columns and "NO2 (ug/m3)" in df_fe.columns:
    df_fe["NO_x_NO2"] = df_fe["NO (ug/m3)"] * df_fe["NO2 (ug/m3)"]

# 3) ratios (avoid div by zero)
if "NO (ug/m3)" in df_fe.columns and "PM2.5 (ug/m3)" in df_fe.columns:
    df_fe["PM25_to_NO"] = df_fe["PM2.5 (ug/m3)"] / (df_fe["NO (ug/m3)"].replace(0, np.nan))

# 4) binning (temperature example)
if "Temp (degree C)" in df_fe.columns:
    df_fe["temp_bin"] = pd.qcut(df_fe["Temp (degree C)"].rank(method="first"), q=3, labels=["low","mid","high"])

print("Feature engineering done; new columns added:", [c for c in df_fe.columns if c.endswith("_sq")][:5])

In [ ]:
# Fixed small autoencoder (safe imputation + scaling + small training)
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1) Prepare numeric-only data safely
numeric_df = df.select_dtypes(include=[np.number]).copy()

# Drop numeric columns that are entirely NaN (no info)
all_nan = numeric_df.columns[numeric_df.isna().all()].tolist()
if all_nan:
    numeric_df = numeric_df.drop(columns=all_nan)
    print("Dropped all-NaN numeric columns:", all_nan)

# If nothing left, raise
if numeric_df.shape[1] == 0:
    raise RuntimeError("No numeric columns available after dropping all-NaN columns.")

# 2) Impute numeric NaNs with column mean (fast)
imp = SimpleImputer(strategy="mean")
X_imputed = imp.fit_transform(numeric_df)    # numpy array, shape (n_samples, n_features)

# 3) Standardize
scaler = StandardScaler()
Xs = scaler.fit_transform(X_imputed)

# 4) Use small subset for speed (adjust as required)
n_train = min(2000, Xs.shape[0])
X_train = Xs[:n_train]

# 5) Import TensorFlow/Keras
try:
    from tensorflow.keras import layers, models, optimizers
except Exception as e:
    raise ImportError("TensorFlow is not available. Install with `pip install -U tensorflow`. Original error: " + str(e))

# 6) Build a tiny autoencoder
input_dim = X_train.shape[1]
encoding_dim = max(2, input_dim // 4)

inp = layers.Input(shape=(input_dim,))
h = layers.Dense(max(encoding_dim * 2, 8), activation="relu")(inp)
encoded = layers.Dense(encoding_dim, activation="relu")(h)
decoded = layers.Dense(input_dim, activation="linear")(encoded)

autoenc = models.Model(inputs=inp, outputs=decoded)
encoder = models.Model(inputs=inp, outputs=encoded)

autoenc.compile(optimizer=optimizers.Adam(1e-3), loss="mse")

# 7) Train (fewer epochs for speed; set verbose=1 if you want progress)
autoenc.fit(X_train, X_train, epochs=20, batch_size=128, validation_split=0.1, verbose=0)

# 8) Extract learned features for all data (predict in batches)
learned_features = encoder.predict(Xs, batch_size=256)
print("Learned features shape:", learned_features.shape)

In [ ]:
print("learned_features shape:", learned_features.shape)
print("First 5 learned feature vectors:\n", learned_features[:5])

In [ ]:
# create column names and add to df (only numeric rows as used)
enc_cols = [f"ae_feat_{i+1}" for i in range(learned_features.shape[1])]
df_enc = df.copy()
# If you used numeric_df earlier: numeric_df.index aligns with learned_features rows.
# Here we'll assume numeric_df index matches df rows; if not, align appropriately.
df_enc.loc[:, enc_cols] = learned_features
print("Added learned features to df_enc. New columns:", enc_cols)

In [ ]:
# get reconstructions from autoencoder
reconstructed = autoenc.predict(Xs, batch_size=256)   # same scale as Xs
mse_per_sample = np.mean((Xs - reconstructed)**2, axis=1)
print("Reconstruction MSE - summary:")
print("min:", mse_per_sample.min(), "median:", np.median(mse_per_sample), "max:", mse_per_sample.max())
# Add to DataFrame if desired
df_enc["ae_recon_mse"] = mse_per_sample

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# TensorFlow / Keras
try:
    from tensorflow.keras import layers, models, optimizers
except Exception as e:
    raise ImportError("TensorFlow/Keras not available. Install with `pip install -U tensorflow`. " + str(e))


N_TRAIN_MAX = 2000        # subset size for training
EPOCHS = 20               # training epochs
BATCH_SIZE = 128
VALIDATION_SPLIT = 0.1
ENCODING_DIV = 4          # encoding_dim = max(2, input_dim // ENCODING_DIV)
TARGET_COL_GUESS = "PM2.5 (ug/m3)"  # optional: used for correlation with learned features
SAVE_PLOTS = True
OUT_DIR = "autoenc_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

numeric_df = df.select_dtypes(include=[np.number]).copy()
if numeric_df.shape[1] == 0:
    raise RuntimeError("No numeric columns found in df.")

all_nan_cols = numeric_df.columns[numeric_df.isna().all()].tolist()
if all_nan_cols:
    numeric_df = numeric_df.drop(columns=all_nan_cols)
    print("Dropped all-NaN numeric columns:", all_nan_cols)

if numeric_df.shape[1] == 0:
    raise RuntimeError("No numeric columns left after dropping all-NaN columns.")

# Keep a copy of numeric_df index to align results back to df
index = numeric_df.index

# -------------------------
# 2) Impute + scale
# -------------------------
imp = SimpleImputer(strategy="mean")
X_imputed = imp.fit_transform(numeric_df)   # numpy array shape (n_samples, n_features)

scaler = StandardScaler()
Xs = scaler.fit_transform(X_imputed)

print(f"Numeric data shape (samples, features): {Xs.shape}")

# -------------------------
# 3) Build tiny autoencoder
# -------------------------
n_samples, input_dim = Xs.shape
encoding_dim = max(2, input_dim // ENCODING_DIV)

# Build model
inp = layers.Input(shape=(input_dim,))
h = layers.Dense(max(encoding_dim * 2, 8), activation="relu")(inp)
encoded = layers.Dense(encoding_dim, activation="relu")(h)
decoded = layers.Dense(input_dim, activation="linear")(encoded)

autoenc = models.Model(inputs=inp, outputs=decoded, name="tiny_autoencoder")
encoder = models.Model(inputs=inp, outputs=encoded, name="tiny_encoder")

autoenc.compile(optimizer=optimizers.Adam(1e-3), loss="mse")

print(f"Autoencoder input_dim={input_dim}, encoding_dim={encoding_dim}")

# -------------------------
# 4) Train on subset for speed
# -------------------------
n_train = min(N_TRAIN_MAX, n_samples)
X_train = Xs[:n_train]

history = autoenc.fit(
    X_train, X_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    verbose=0  # set to 1 if you want epoch logs
)

print(f"Trained autoencoder for {EPOCHS} epochs on {n_train} samples (subset).")

learned_features = encoder.predict(Xs, batch_size=256)
reconstructed = autoenc.predict(Xs, batch_size=256)

print("Learned features shape:", learned_features.shape)

# Reconstruction MSE per sample (in standardized space)
mse_per_sample = np.mean((Xs - reconstructed) ** 2, axis=1)
print("Reconstruction MSE (summary):",
      f"min={mse_per_sample.min():.6f}, median={np.median(mse_per_sample):.6f}, max={mse_per_sample.max():.6f}")

ae_cols = [f"ae_feat_{i+1}" for i in range(learned_features.shape[1])]
df_ae = df.copy()

ae_df = pd.DataFrame(learned_features, columns=ae_cols, index=index)
ae_df_recon = pd.DataFrame({"ae_recon_mse": mse_per_sample}, index=index)

df_ae = pd.concat([df_ae, ae_df, ae_df_recon], axis=1)
print(f"Added {len(ae_cols)} learned feature columns and reconstruction MSE to df_ae.")


print("\nFirst 3 learned feature vectors (rows):")
print(ae_df.head(3))

# Top rows by reconstruction error (possible outliers)
top_err = ae_df_recon["ae_recon_mse"].sort_values(ascending=False).head(5)
print("\nTop 5 rows by reconstruction MSE (possible outliers):")
print(top_err)

# Correlate learned features with an available numeric target, if present
if TARGET_COL_GUESS in numeric_df.columns:
    target_y = numeric_df[TARGET_COL_GUESS]
    corr_with_target = ae_df.corrwith(target_y).abs().sort_values(ascending=False)
    print(f"\nAbsolute correlations between AE features and target '{TARGET_COL_GUESS}':")
    print(corr_with_target)
else:
    print(f"\nTarget '{TARGET_COL_GUESS}' not found among numeric columns; skipping AE->target correlations.")

# -------------------------
# 8) PCA visualization of latent space (2D)
# -------------------------
if learned_features.shape[1] >= 2:
    pca_vis = PCA(n_components=2).fit_transform(learned_features)
    plt.figure(figsize=(6,5))
    plt.scatter(pca_vis[:,0], pca_vis[:,1], s=6, alpha=0.6)
    plt.title("PCA projection of AE latent space (2D)")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.tight_layout()
    if SAVE_PLOTS:
        p = os.path.join(OUT_DIR, "ae_latent_pca2d.png")
        plt.savefig(p, dpi=200, bbox_inches="tight")
        print("Saved latent PCA plot to:", p)
    plt.show()

# -------------------------
# 9) Histogram of reconstruction error
# -------------------------
plt.figure(figsize=(6,4))
sns.histplot(mse_per_sample, bins=50, kde=False)
plt.title("Reconstruction MSE distribution")
plt.xlabel("MSE")
plt.tight_layout()
if SAVE_PLOTS:
    p = os.path.join(OUT_DIR, "ae_recon_mse_hist.png")
    plt.savefig(p, dpi=200, bbox_inches="tight")
    print("Saved reconstruction MSE histogram to:", p)
plt.show()

# -------------------------
# 10) Save encoder weights (optional)
# -------------------------
encoder_path = os.path.join(OUT_DIR, "ae_encoder.h5")
try:
    encoder.save(encoder_path)
    print("Saved encoder model to:", encoder_path)
except Exception as e:
    print("Could not save encoder model:", e)

# -------------------------
# Quick usage examples
# -------------------------
print("\nQuick usage examples:")
print(f"- Encoded data shape: {learned_features.shape}")
print(f"- To use learned features in ML: X_ae = ae_df.values  (shape {ae_df.shape})")
print(f"- Reconstruction MSE is in df_ae['ae_recon_mse']")

# Optionally return objects in interactive session (if running in notebook)
try:
    # expose a few objects to interactive namespace
    globals().update({
        "ae_df": ae_df,
        "ae_recon_mse": ae_df_recon,
        "learned_features": learned_features,
        "encoder_model": encoder,
        "autoencoder_model": autoenc,
        "scaler": scaler,
        "imputer": imp
    })
except Exception:
    pass

print("\nDone.")